# 3 Neural Machine Translation

**Neural machine translation (NMT):** $\;$ original application of Transformer

**AT25:** $\;$ AT25 is based on PyTorch; we set the seed for random number generation

In [4]:
import torch; import torch.nn as nn; torch.manual_seed(23)
import import_ipynb; from at251 import create_model, create_causal_mask

**Model:** $\;$ simple random model

In [5]:
model = create_model(src_vocab_size=3, tgt_vocab_size=3, embed_dim=2, num_layers=1, num_heads=1, dropout=0.)

**Input text embedding:** $\;$ sequence of `embed_dim`-dimensional arrays

In [6]:
src = torch.LongTensor([[1, 2, 1]]); model.src_embed(src).data

tensor([[[ 0.0415,  2.2411],
         [ 1.2927, -0.6469],
         [ 0.9508,  0.8249]]])

**Output text embedding:** $\;$ sequence of `embed_dim`-dimensional arrays

In [7]:
tgt = torch.LongTensor([[0]]); model.tgt_embed(tgt).data

tensor([[[1.3548, 1.4321]]])

**Final linear projection:** $\;$ `generator` transform the decoder output into the probablistic prediction of the model

In [8]:
model.generator.final_proj.weight = nn.Parameter(torch.randn_like(model.generator.final_proj.weight))
model.generator.final_proj(torch.tensor([[[-0.7071,  0.7071]]])).data

tensor([[[-0.0080, -0.3012, -0.0413]]])


<p style="page-break-after:always;"></p>


**Evaluation:**

In [9]:
model.eval()
src = torch.LongTensor([[1, 2, 1]]); print(f"Input text: {src}")
src_mask = torch.ones(1, 1, 3); print(f"Input mask: {src_mask}\n")
memory = model.encode(src, src_mask)
ys = torch.zeros(1, 1).type_as(src) # note: ys[0]=0, i.e., ys starts with 0
for i in range(2): 
  print(f"Inference of the next token after having predicted {i} tokens:") 
  print(f"* Already predicted output text: {ys}") 
  tgt_mask = create_causal_mask(ys.size(1)).type_as(src.data) 
  print(f"* Output mask: {str(tgt_mask).replace('\n','')}") 
  out = model.decode(ys, memory, src_mask, tgt_mask) 
  print(f"* Decoder output: {str(out.data).replace('\n','')}") 
  prob = model.generator(out[:, -1]) # last token 
  print(f"* Logsoftmax of unembedding of the last token in the output: {str(prob.data).replace('\n','')}") 
  _, next_word = torch.max(prob, dim=1) 
next_word = next_word.data[0] 
ys = torch.cat([ys, torch.empty(1, 1).type_as(src.data).fill_(next_word)], dim=1) 
print("* Output text with new inferred token:", ys, "\n")

Input text: tensor([[1, 2, 1]])
Input mask: tensor([[[1., 1., 1.]]])

Inference of the next token after having predicted 0 tokens:
* Already predicted output text: tensor([[0]])
* Output mask: tensor([[[1]]])
* Decoder output: tensor([[[-0.7071,  0.7071]]])
* Logsoftmax of unembedding of the last token in the output: tensor([[-0.9981, -1.2913, -1.0314]])
Inference of the next token after having predicted 1 tokens:
* Already predicted output text: tensor([[0]])
* Output mask: tensor([[[1]]])
* Decoder output: tensor([[[-0.7071,  0.7071]]])
* Logsoftmax of unembedding of the last token in the output: tensor([[-0.9981, -1.2913, -1.0314]])
* Output text with new inferred token: tensor([[0, 0]]) 




<p style="page-break-after:always;"></p>
